In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL DO AMBIENTE
# ============================================================
#
# Nesta etapa, preparamos o ambiente para que o Spark consiga interagir com o S3 
# utilizando as credenciais temporárias do Learner Lab.
#
# Pontos de configuração esperados no .env:
#
#   AWS_ACCESS_KEY_ID      Chave de acesso temporária da AWS.
#   AWS_SECRET_ACCESS_KEY  Chave secreta temporária da AWS.
#   AWS_SESSION_TOKEN      Token temporário da sessão AWS Academy.
#   S3_BUCKET              Nome do bucket S3 do integrante.
#
# ============================================================
# BIBLIOTECAS
# ============================================================

# Setup Env: Dotenv/Pathlib
import os
import sys
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

# ============================================================
# CAMINHOS DO PROJETO E VARIÁVEIS DE AMBIENTE
# ============================================================

# Localiza o arquivo .env, carrega suas variáveis e define a raiz do projeto.
load_dotenv(find_dotenv())
PROJECT_ROOT = Path(find_dotenv()).parent

# ============================================================
# CONFIGURAÇÃO DO PYSPARK
# ============================================================

# Garante que o PySpark use o mesmo interpretador Python do ambiente atual.
# Isso evita conflitos comuns em WSL, Conda ou ambientes virtuais.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# ============================================================
# CREDENCIAIS AWS E BUCKET S3
# ============================================================

# Carrega as credenciais temporárias e o bucket S3 definidos no .env.
AWS_ACCESS_KEY_ID     = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN     = os.getenv("AWS_SESSION_TOKEN")
S3_BUCKET             = os.getenv("S3_BUCKET")

# Interrompe a execução se alguma configuração obrigatória não foi carregada.
if not all([AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_SESSION_TOKEN, S3_BUCKET]):
    raise EnvironmentError("Falha ao carregar credenciais AWS ou S3_BUCKET do arquivo .env")

# ============================================================
# CONFERÊNCIA DA CONFIGURAÇÃO
# ============================================================

print(f"Projeto: {PROJECT_ROOT}\nBucket : {S3_BUCKET}")

Projeto: /mnt/d/diego/01_projects/postech-challenge-2
Bucket : alfabetizacao-data-lake-diego


In [2]:
# ============================================================
# CRIAÇÃO DA SESSÃO SPARK COM ACESSO AO S3
# ============================================================
#
# Esta célula cria a SparkSession usada para gravar os resultados da camada Silver no S3.
#
#
# ============================================================
# BIBLIOTECAS
# ============================================================

# SparkSession: S3 Config
from pyspark.sql import SparkSession

# ============================================================
# SESSÃO SPARK
# ============================================================

spark = (
    SparkSession.builder
    .appName("transformar-e-armazenar-S3")
    .master("local[*]") # remover quando for implementar a computação em nuvem
    # Configuração de Pacotes e S3A
    .config("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.3.4,""com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY_ID)
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.session.token", AWS_SESSION_TOKEN)
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com")
    .getOrCreate()
)

# ============================================================
# CONFERÊNCIA DA SESSÃO
# ============================================================

print(f"Sessão Spark {spark.version} criada com sucesso.")

26/07/08 21:07:40 WARN Utils: Your hostname, lua resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/08 21:07:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/diego/miniconda3/envs/postech2/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/diego/.ivy2/cache
The jars for the packages stored in: /home/diego/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e143cfac-bf7a-4a4d-8087-0cd3252c785d;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 363ms :: artifacts dl 14ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evict

Sessão Spark 3.5.5 criada com sucesso.


In [3]:
# ============================================================
# CLIENTE S3 PARA OPERAÇÕES ADMINISTRATIVAS
# ============================================================
#
# Esta célula cria um cliente S3 com boto3 usando as mesmas
# credenciais temporárias carregadas do arquivo .env.
#
# ============================================================
# BIBLIOTECAS
# ============================================================

# Boto3: Gestão de pastas
import boto3

# ============================================================
# CLIENTE S3
# ============================================================

# Inicializa o cliente S3 logo após o setup das credenciais.
s3_client = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN
)

In [4]:
from datetime import datetime, timezone
from pyspark.sql import functions as F

SILVER_BASE = f"s3a://{S3_BUCKET}/silver"
GOLD_BASE = f"s3a://{S3_BUCKET}/gold"
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

GOLD_TS   = datetime.now(timezone.utc).isoformat()   # timestamp único do batch, como SILVER_TS
def add_gold_metadata(df):
    return df.withColumn("_gold_processed_at", F.lit(GOLD_TS))

### Consolidação da Camada Gold: Indicadores Municipais
Nesta etapa, realizamos o cruzamento final entre os resultados de alfabetização e as metas projetadas para os municípios. A lógica principal utiliza CTEs (Common Table Expressions) para transformar a estrutura das metas de colunas para linhas (unpivot via stack), garantindo que cada ano de projeção (2024-2030) seja comparável com os resultados anuais.



In [11]:
spark.read.parquet(f"{SILVER_BASE}/municipio").createOrReplaceTempView("silver_municipio")
spark.read.parquet(f"{SILVER_BASE}/meta_alfabetizacao_municipio").createOrReplaceTempView("silver_meta_municipio")

gold_ind_mun = add_gold_metadata(spark.sql("""
WITH
-- (1) META -> LONG + COALESCE latest-non-null: 1 linha por (municipio, ano-alvo)
meta_long_raw AS (
    SELECT ano AS ano_pub, id_municipio,
           stack(7,
             2024, meta_alfabetizacao_2024, 2025, meta_alfabetizacao_2025,
             2026, meta_alfabetizacao_2026, 2027, meta_alfabetizacao_2027,
             2028, meta_alfabetizacao_2028, 2029, meta_alfabetizacao_2029,
             2030, meta_alfabetizacao_2030
           ) AS (ano, meta)
    FROM silver_meta_municipio
),
meta_long AS (
    SELECT id_municipio, ano, meta FROM (
        SELECT *, ROW_NUMBER() OVER (
            PARTITION BY id_municipio, ano
            ORDER BY (meta IS NOT NULL) DESC, ano_pub DESC   -- não-nulo, depois publicação mais nova
        ) rn FROM meta_long_raw
    ) WHERE rn = 1 AND meta IS NOT NULL
),
-- (2) RESULTADO no nível da meta (rede=3, Municipal) — fonte canônica da taxa
res AS (
    SELECT ano, id_municipio, nome_municipio, id_uf, sigla_uf,
           taxa_alfabetizacao, media_portugues
    FROM silver_municipio
    WHERE rede = 3
),
-- (3) PARTICIPAÇÃO (por ano de avaliação) — só existe na tabela de meta
part AS (
    SELECT ano, id_municipio, percentual_participacao
    FROM silver_meta_municipio
)
SELECT
    r.ano,
    r.id_municipio,
    r.nome_municipio,
    r.id_uf,
    r.sigla_uf,
    3                                        AS rede,   -- explícito: comparação no nível Municipal
    r.taxa_alfabetizacao,
    r.media_portugues,
    p.percentual_participacao,
    ml.meta,
    ROUND(r.taxa_alfabetizacao - ml.meta, 2)                        AS distancia_meta,
    CASE WHEN ml.meta IS NULL THEN NULL
         ELSE (r.taxa_alfabetizacao - ml.meta) >= 0 END            AS atingiu_meta,
    CASE WHEN ml.meta IS NULL                      THEN NULL
         WHEN r.taxa_alfabetizacao - ml.meta >= 5  THEN 'Muito acima'
         WHEN r.taxa_alfabetizacao - ml.meta >= 0  THEN 'Acima'
         WHEN r.taxa_alfabetizacao - ml.meta >= -5 THEN 'Próximo'
         ELSE 'Muito abaixo' END                                   AS categoria_desempenho,
    CASE WHEN p.percentual_participacao IS NULL THEN NULL
         WHEN p.percentual_participacao >= 95   THEN 'Alta'
         WHEN p.percentual_participacao >= 80   THEN 'Média'
         ELSE 'Baixa' END                                          AS faixa_participacao
FROM res r
LEFT JOIN part      p  ON r.ano = p.ano AND r.id_municipio = p.id_municipio
LEFT JOIN meta_long ml ON r.ano = ml.ano AND r.id_municipio = ml.id_municipio
"""))

(gold_ind_mun.write.mode("overwrite").partitionBy("ano")
    .parquet(f"{GOLD_BASE}/indicadores_municipio"))
print("gold/indicadores_municipio:", gold_ind_mun.count())
gold_ind_mun.orderBy("ano","id_municipio").show(8, truncate=False)

gold/indicadores_municipio: 16396


+----+------------+---------------------+-----+--------+----+------------------+---------------+-----------------------+----+--------------+------------+--------------------+------------------+--------------------------------+
|ano |id_municipio|nome_municipio       |id_uf|sigla_uf|rede|taxa_alfabetizacao|media_portugues|percentual_participacao|meta|distancia_meta|atingiu_meta|categoria_desempenho|faixa_participacao|_gold_processed_at              |
+----+------------+---------------------+-----+--------+----+------------------+---------------+-----------------------+----+--------------+------------+--------------------+------------------+--------------------------------+
|2023|1100015     |Alta Floresta D'Oeste|11   |RO      |3   |64.55             |758.3304       |89.4                   |NULL|NULL          |NULL        |NULL                |Média             |2026-07-09T00:08:01.614965+00:00|
|2023|1100023     |Ariquemes            |11   |RO      |3   |62.3              |757.0999    

### Consolidação da Camada Gold: Indicadores Estaduais (UF)
Nesta etapa, consolidamos os indicadores de desempenho em nível estadual (UF) na camada Gold. O processo realiza o cruzamento entre os resultados reais de alfabetização e as metas projetadas, utilizando uma transformação de unpivot (via stack) para alinhar as metas anuais (2024-2030) com os dados de séries históricas.

O foco desta análise é a rede pública (rede = 5), permitindo calcular métricas de sucesso como a distância em relação à meta, o status de atingimento e classificações qualitativas de desempenho e participação. Os dados resultantes são enriquecidos com metadados e persistidos no S3, servindo como base para dashboards executivos de nível estadual.

In [18]:
spark.read.parquet(f"{SILVER_BASE}/uf").createOrReplaceTempView("silver_uf")
spark.read.parquet(f"{SILVER_BASE}/meta_alfabetizacao_uf").createOrReplaceTempView("silver_meta_uf")

gold_ind_uf = add_gold_metadata(spark.sql("""
WITH
meta_long_raw AS (
    SELECT ano AS ano_pub, sigla_uf,
           stack(7,
             2024, meta_alfabetizacao_2024, 2025, meta_alfabetizacao_2025,
             2026, meta_alfabetizacao_2026, 2027, meta_alfabetizacao_2027,
             2028, meta_alfabetizacao_2028, 2029, meta_alfabetizacao_2029,
             2030, meta_alfabetizacao_2030
           ) AS (ano, meta)
    FROM silver_meta_uf
),
meta_long AS (
    SELECT sigla_uf, ano, meta FROM (
        SELECT *, ROW_NUMBER() OVER (
            PARTITION BY sigla_uf, ano
            ORDER BY (meta IS NOT NULL) DESC, ano_pub DESC
        ) rn FROM meta_long_raw
    ) WHERE rn = 1 AND meta IS NOT NULL
),
res AS (
    SELECT ano, id_uf, sigla_uf, taxa_alfabetizacao, media_portugues
    FROM silver_uf
    WHERE rede = 5
),
part AS (
    SELECT ano, sigla_uf, percentual_participacao
    FROM silver_meta_uf
)
SELECT
    r.ano,
    r.id_uf,
    r.sigla_uf,
    5                                        AS rede,   -- comparação no nível Pública
    r.taxa_alfabetizacao,
    r.media_portugues,
    p.percentual_participacao,
    ml.meta,
    ROUND(r.taxa_alfabetizacao - ml.meta, 2)                        AS distancia_meta,
    CASE WHEN ml.meta IS NULL THEN NULL
         ELSE (r.taxa_alfabetizacao - ml.meta) >= 0 END            AS atingiu_meta,
    CASE WHEN ml.meta IS NULL                      THEN NULL
         WHEN r.taxa_alfabetizacao - ml.meta >= 5  THEN 'Muito acima'
         WHEN r.taxa_alfabetizacao - ml.meta >= 0  THEN 'Acima'
         WHEN r.taxa_alfabetizacao - ml.meta >= -5 THEN 'Próximo'
         ELSE 'Muito abaixo' END                                   AS categoria_desempenho,
    CASE WHEN p.percentual_participacao IS NULL THEN NULL
         WHEN p.percentual_participacao >= 95   THEN 'Alta'
         WHEN p.percentual_participacao >= 80   THEN 'Média'
         ELSE 'Baixa' END                                          AS faixa_participacao
FROM res r
LEFT JOIN part      p  ON r.ano = p.ano AND r.sigla_uf = p.sigla_uf
LEFT JOIN meta_long ml ON r.ano = ml.ano AND r.sigla_uf = ml.sigla_uf
"""))

(gold_ind_uf.write.mode("overwrite").partitionBy("ano")
    .parquet(f"{GOLD_BASE}/indicadores_uf"))
print("gold/indicadores_uf:", gold_ind_uf.count())
gold_ind_uf.orderBy("ano","sigla_uf").show(30, truncate=False)

gold/indicadores_uf: 76


+----+-----+--------+----+------------------+---------------+-----------------------+----+--------------+------------+--------------------+------------------+--------------------------------+
|ano |id_uf|sigla_uf|rede|taxa_alfabetizacao|media_portugues|percentual_participacao|meta|distancia_meta|atingiu_meta|categoria_desempenho|faixa_participacao|_gold_processed_at              |
+----+-----+--------+----+------------------+---------------+-----------------------+----+--------------+------------+--------------------+------------------+--------------------------------+
|2023|27   |AL      |5   |43.88             |729.7227       |92.0                   |NULL|NULL          |NULL        |NULL                |Média             |2026-07-09T00:08:01.614965+00:00|
|2023|13   |AM      |5   |52.2              |736.4687       |76.0                   |NULL|NULL          |NULL        |NULL                |Baixa             |2026-07-09T00:08:01.614965+00:00|
|2023|16   |AP      |5   |41.56         

### Consolidação da Camada Gold: Contexto Individual do Aluno
Nesta etapa, criamos uma visão enriquecida dos microdados dos alunos, cruzando os dados individuais da camada Silver com os indicadores de contexto municipal da camada Gold. O objetivo é fornecer uma base consolidada onde cada registro de aluno contém não apenas sua proficiência e status de alfabetização, mas também o desempenho do município onde estuda (taxa de alfabetização local, distância da meta e categoria de desempenho).

Principais Transformações:

- Herança de Contexto: Atributos do município são prefixados com ctx_ para diferenciar das métricas individuais.  
- Cálculo de Gap: Criação da métrica gap_proficiencia, calculando a distância em relação ao ponto de corte (743).  
- Definição de Alvo (Label): O campo alfabetizado é mapeado como label_alfabetizado para fins de treinamento.  
- Filtro de Qualidade: Remoção de registros onde a proficiência é nula, garantindo que a base final contenha apenas estudantes efetivamente avaliados (aproximadamente 5,32 milhões de registros).

In [21]:
spark.read.parquet(f"{SILVER_BASE}/alunos").createOrReplaceTempView("silver_alunos")
spark.read.parquet(f"{GOLD_BASE}/indicadores_municipio").createOrReplaceTempView("gold_ind_mun")

gold_aluno_ctx = add_gold_metadata(spark.sql("""
    SELECT
        -- identificadores (não-features)
        a.id_aluno,
        a.ano,
        a.id_municipio,
        SUBSTRING(a.id_municipio, 1, 2)  AS id_uf,   -- derivado do IBGE (sempre presente); sigla_uf omitida
        -- atributos do aluno
        a.rede,
        a.caderno,
        -- contexto do município (features de lugar, herdadas da Gold municipal)
        c.taxa_alfabetizacao        AS ctx_taxa_municipio,
        c.media_portugues           AS ctx_media_municipio,
        c.meta                      AS ctx_meta_municipio,
        c.distancia_meta            AS ctx_distancia_meta_municipio,
        c.atingiu_meta              AS ctx_atingiu_meta_municipio,
        c.percentual_participacao   AS ctx_participacao_municipio,
        c.categoria_desempenho      AS ctx_categoria_municipio,
        c.faixa_participacao        AS ctx_faixa_participacao_municipio,
        -- medição do aluno
        a.proficiencia,
        ROUND(a.proficiencia - 743, 2) AS gap_proficiencia,   -- proficiencia - corte
        -- ALVO
        a.alfabetizado              AS label_alfabetizado
    FROM silver_alunos a
    LEFT JOIN gold_ind_mun c
      ON a.ano = c.ano AND a.id_municipio = c.id_municipio
    WHERE a.proficiencia IS NOT NULL
"""))

(gold_aluno_ctx.write.mode("overwrite").partitionBy("ano")
    .parquet(f"{GOLD_BASE}/aluno_contexto"))
print("gold/aluno_contexto:", gold_aluno_ctx.count())   # ~5,32M
gold_aluno_ctx.show(5, truncate=False)

26/07/08 21:48:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/07/08 21:48:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/07/08 21:48:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/07/08 21:48:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/07/08 21:48:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/07/08 21:48:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/07/08 21:48:52 WARN MemoryManager: Total allocation exceeds 95.

gold/aluno_contexto: 5321266


+--------+----+------------+-----+----+-------+------------------+-------------------+------------------+----------------------------+--------------------------+--------------------------+-----------------------+--------------------------------+------------+----------------+------------------+--------------------------------+
|id_aluno|ano |id_municipio|id_uf|rede|caderno|ctx_taxa_municipio|ctx_media_municipio|ctx_meta_municipio|ctx_distancia_meta_municipio|ctx_atingiu_meta_municipio|ctx_participacao_municipio|ctx_categoria_municipio|ctx_faixa_participacao_municipio|proficiencia|gap_proficiencia|label_alfabetizado|_gold_processed_at              |
+--------+----+------------+-----+----+-------+------------------+-------------------+------------------+----------------------------+--------------------------+--------------------------+-----------------------+--------------------------------+------------+----------------+------------------+--------------------------------+
|31187261|2025|3

In [ ]:
# ============================================================
# DICIONÁRIO DE PAPÉIS — gold/aluno_contexto (consumido pelo pipeline de ML)
# ============================================================
PAPEIS_ALUNO_CONTEXTO = {
    "alvo":         ["label_alfabetizado"],
    "identificador":["id_aluno", "ano", "id_municipio", "id_uf"],
    "vazamento":    ["proficiencia", "gap_proficiencia"],   # DERIVAM do alvo -> NUNCA como feature
    "constante":    [],   # presenca/preenchimento não entraram (=1 em todos os medidos)
    "features":     [
        "rede", "caderno",
        "ctx_taxa_municipio", "ctx_media_municipio", "ctx_meta_municipio",
        "ctx_distancia_meta_municipio", "ctx_atingiu_meta_municipio",
        "ctx_participacao_municipio", "ctx_categoria_municipio",
        "ctx_faixa_participacao_municipio",
    ],
}
# No treino: X = df.select(PAPEIS_ALUNO_CONTEXTO["features"]); y = df["label_alfabetizado"]

### Checar Qualidade

In [ ]:
# ============================================================
# VALIDAÇÃO DE QUALIDADE — CAMADA SILVER
# ============================================================
# Espelha etl-silver.py: catálogo declarativo (CHECKS) + severidade
# por regra (critico) -> PASS/FAIL/WARN, Score e raise em falha crítica.
# Tipos: min_count, not_null, unique (aceita chave composta), range.
# (No .py a saída vai por logging; aqui usamos print p/ mostrar inline.)

def checar_qualidade(entidade, df, checks):
    print(f"[DQ:GOLD] {entidade} | iniciando | checks={len(checks)}")
    passou = falhou = criticos = 0

    for check in checks:
        tipo    = check["tipo"]
        coluna  = check.get("coluna")
        valor   = check.get("valor")
        critico = check.get("critico", True)
        ok, detalhe = False, ""

        try:
            if tipo == "min_count":
                n  = df.count()
                ok, detalhe = n >= valor, f"contagem={n} | minimo={valor}"
            elif tipo == "not_null":
                nulos = df.filter(F.col(coluna).isNull()).count()
                ok, detalhe = nulos == 0, f"{nulos} nulos"
            elif tipo == "unique":
                cols = coluna if isinstance(coluna, list) else [coluna]
                dups = df.count() - df.select(*cols).distinct().count()
                ok, detalhe = dups == 0, f"{dups} duplicatas (chave={cols})"
            elif tipo == "range":
                mn, mx = valor
                fora = df.filter((F.col(coluna) < mn) | (F.col(coluna) > mx)).count()
                ok, detalhe = fora == 0, f"{fora} fora de [{mn},{mx}]"
            elif tipo == "expr":                                  # <-- ramo que faltava
                viol = df.filter(valor).count()                  # valor = SQL da VIOLAÇÃO
                ok, detalhe = viol == 0, f"{viol} linhas violam: {valor}"
        except Exception as e:
            ok, detalhe = False, f"Erro: {e}"

        status = "PASS" if ok else ("FAIL" if critico else "WARN")
        print(f"[DQ:GOLD] {status:4} | {tipo:9} | {coluna if coluna else '-'} | {detalhe}")
        if ok: passou += 1
        else:
            falhou += 1
            criticos += 1 if critico else 0

    score = round(passou / len(checks) * 100, 1)
    print(f"[DQ:GOLD] {entidade} | Score={score}% | PASS={passou} FAIL={falhou}\n")
    if criticos > 0:
        raise Exception(f"[DQ:GOLD] {entidade}: {criticos} check(s) critico(s) falharam. Pipeline interrompido.")
    return score


# No checar_qualidade (copiada do Silver), adicione:
#     elif tipo == "expr":
#         viol = df.filter(valor).count()      # valor = SQL que descreve a VIOLAÇÃO
#         ok, detalhe = viol == 0, f"{viol} linhas violam: {valor}"

CHECKS_GOLD = {
    "indicadores_municipio": [
        {"tipo":"min_count","valor":1,                                        "critico":True},
        {"tipo":"not_null", "coluna":"ano",                                   "critico":True},
        {"tipo":"not_null", "coluna":"id_municipio",                          "critico":True},
        {"tipo":"unique",   "coluna":["ano","id_municipio"],                 "critico":True},
        {"tipo":"range",    "coluna":"taxa_alfabetizacao","valor":(0,100),    "critico":False},
        {"tipo":"range",    "coluna":"meta","valor":(0,100),                 "critico":False},
        {"tipo":"range",    "coluna":"percentual_participacao","valor":(0,100),"critico":False},
        # regra de negócio: atingiu_meta coerente com (taxa - meta)
        {"tipo":"expr","coluna":"atingiu_meta",
         "valor":"meta IS NOT NULL AND ((taxa_alfabetizacao - meta >= 0) <> atingiu_meta)","critico":True},
    ],
}

for tabela, checks in CHECKS_GOLD.items():
    df = spark.read.parquet(f"{GOLD_BASE}/{tabela}")
    checar_qualidade(tabela, df, checks)

[DQ:GOLD] indicadores_municipio | iniciando | checks=8


[DQ:GOLD] PASS | min_count | - | contagem=16396 | minimo=1
[DQ:GOLD] PASS | not_null  | ano | 0 nulos


[DQ:GOLD] PASS | not_null  | id_municipio | 0 nulos


[DQ:GOLD] PASS | unique    | ['ano', 'id_municipio'] | 0 duplicatas (chave=['ano', 'id_municipio'])


[DQ:GOLD] PASS | range     | taxa_alfabetizacao | 0 fora de [0,100]


[DQ:GOLD] PASS | range     | meta | 0 fora de [0,100]


[DQ:GOLD] PASS | range     | percentual_participacao | 0 fora de [0,100]


[DQ:GOLD] PASS | expr      | atingiu_meta | 0 linhas violam: meta IS NOT NULL AND ((taxa_alfabetizacao - meta >= 0) <> atingiu_meta)
[DQ:GOLD] indicadores_municipio | Score=100.0% | PASS=8 FAIL=0



In [13]:
CHECKS_GOLD["indicadores_uf"] = [
    {"tipo":"min_count","valor":1,                                         "critico":True},
    {"tipo":"not_null", "coluna":"ano",                                    "critico":True},
    {"tipo":"not_null", "coluna":"sigla_uf",                              "critico":True},
    {"tipo":"unique",   "coluna":["ano","sigla_uf"],                      "critico":True},
    {"tipo":"range",    "coluna":"taxa_alfabetizacao","valor":(0,100),     "critico":False},
    {"tipo":"range",    "coluna":"meta","valor":(0,100),                  "critico":False},
    {"tipo":"range",    "coluna":"percentual_participacao","valor":(0,100),"critico":False},
    {"tipo":"expr","coluna":"atingiu_meta",
     "valor":"meta IS NOT NULL AND ((taxa_alfabetizacao - meta >= 0) <> atingiu_meta)","critico":True},
]

df = spark.read.parquet(f"{GOLD_BASE}/indicadores_uf")
checar_qualidade("indicadores_uf", df, CHECKS_GOLD["indicadores_uf"])

[DQ:GOLD] indicadores_uf | iniciando | checks=8


[DQ:GOLD] PASS | min_count | - | contagem=76 | minimo=1
[DQ:GOLD] PASS | not_null  | ano | 0 nulos


[DQ:GOLD] PASS | not_null  | sigla_uf | 0 nulos


[DQ:GOLD] PASS | unique    | ['ano', 'sigla_uf'] | 0 duplicatas (chave=['ano', 'sigla_uf'])


[DQ:GOLD] PASS | range     | taxa_alfabetizacao | 0 fora de [0,100]


[DQ:GOLD] PASS | range     | meta | 0 fora de [0,100]


[DQ:GOLD] PASS | range     | percentual_participacao | 0 fora de [0,100]


[DQ:GOLD] PASS | expr      | atingiu_meta | 0 linhas violam: meta IS NOT NULL AND ((taxa_alfabetizacao - meta >= 0) <> atingiu_meta)
[DQ:GOLD] indicadores_uf | Score=100.0% | PASS=8 FAIL=0



100.0

In [22]:
CHECKS_GOLD["aluno_contexto"] = [
    {"tipo":"min_count","valor":1,                                   "critico":True},
    {"tipo":"not_null", "coluna":"id_aluno",                        "critico":True},
    {"tipo":"not_null", "coluna":"ano",                             "critico":True},
    {"tipo":"not_null", "coluna":"proficiencia",                    "critico":True},   # filtro garante
    {"tipo":"not_null", "coluna":"label_alfabetizado",             "critico":True},   # sem NULL no alvo
    {"tipo":"unique",   "coluna":["ano","id_aluno"],               "critico":True},
    {"tipo":"range",    "coluna":"proficiencia","valor":(0,1000),   "critico":False},
    # regra de negócio: alvo coerente com o corte 743
    {"tipo":"expr","coluna":"label_alfabetizado",
     "valor":"(proficiencia >= 743) <> (label_alfabetizado = 1)",  "critico":True},
]

df = spark.read.parquet(f"{GOLD_BASE}/aluno_contexto")
checar_qualidade("aluno_contexto", df, CHECKS_GOLD["aluno_contexto"])

[DQ:GOLD] aluno_contexto | iniciando | checks=8


[DQ:GOLD] PASS | min_count | - | contagem=5321266 | minimo=1


[DQ:GOLD] PASS | not_null  | id_aluno | 0 nulos
[DQ:GOLD] PASS | not_null  | ano | 0 nulos


[DQ:GOLD] PASS | not_null  | proficiencia | 0 nulos


[DQ:GOLD] PASS | not_null  | label_alfabetizado | 0 nulos


26/07/08 21:51:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/08 21:51:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/08 21:51:47 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/08 21:51:47 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/08 21:51:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/08 21:51:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/08 21:51:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/08 21:51:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/08 21:51:54 WARN RowBasedKeyValueBatch: Calling spill() on

[DQ:GOLD] PASS | unique    | ['ano', 'id_aluno'] | 0 duplicatas (chave=['ano', 'id_aluno'])


[DQ:GOLD] PASS | range     | proficiencia | 0 fora de [0,1000]


[DQ:GOLD] PASS | expr      | label_alfabetizado | 0 linhas violam: (proficiencia >= 743) <> (label_alfabetizado = 1)
[DQ:GOLD] aluno_contexto | Score=100.0% | PASS=8 FAIL=0



100.0